# Heart Disease Risk Prediction System

**Educational research prototype — a screening aid, not a medical device**

**Author:** Ar Kar Phyo

---

### 1. What this notebook does

This notebook is the complete implementation of the Heart Disease Risk Prediction System. I built it as one end-to-end pipeline that goes from a raw CSV file to a working, interactive AI system:

| Stage | Where it appears below |
|---|---|
| Data acquisition, cleaning and exploratory analysis | Steps 1–2 |
| Preprocessing: encoding, scaling, train/test split | Step 3 |
| Training three classifiers and comparing them | Step 4 |
| Evaluation: confusion matrix, accuracy, precision, recall, F1 | Step 5 |
| Predicting a single new patient + API-generated explanation | Steps 6 & 6b |
| Responsible AI reflection | Step 7 |
| Model packaging (joblib) + interactive demo with Streamlit | Steps 8–9 |

**Final deliverable:** a working AI system that classifies a patient as *at risk* or *not at risk*
of heart disease from 11 clinical features, with an interactive demo in the Streamlit web app — plus an
**API layer**: the official Kaggle API fetches my data, and the Google Gemini API (free tier) writes
plain-language explanations of each prediction.

### 2. How I run it

1. I open this notebook in **Google Colab** (or in **VS Code**) and select **Run all**.
2. Step 1 fetches **`heart.csv`** (*Heart Failure Prediction* by fedesoriano,
   https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) **through the official Kaggle
   API** — it asks me once for my free Kaggle token (an access token from kaggle.com → Settings →
   API Tokens). If I skip that, a file-picker fallback loads the same file downloaded by hand.
3. Step 6b asks once for my free **Gemini API key** (created at aistudio.google.com) and uses the
   `gemini-3.5-flash-lite` model
   to explain each prediction in plain language; leaving it empty keeps the notebook fully offline
   with the built-in explanation.
4. Step 8 packs the best model into **`heart_model.pkl`** and Step 9 writes **`app.py`**, my
   **Streamlit** web app. I launch it from a terminal with `streamlit run app.py` (in VS Code)
   and it opens on **http://localhost:8501** — sliders, Predict button and model-card sidebar
   for my live presentation. Colab and Streamlit-Cloud options are described in the last cell.


## Setup — the libraries I use

I use NumPy and pandas for data handling, Matplotlib and Seaborn for visualisation, and
scikit-learn for preprocessing, modelling and evaluation. I also fix a random seed
(`RANDOM_STATE`) at the top so that every split and every model I train is reproducible —
anyone who re-runs this notebook gets my exact same numbers. My two API integrations also stay light:
the Kaggle API ships as a small pip package installed inside its own cell, and the Gemini API in Step 6b
is called with nothing but Python's built-in `urllib` — no extra setup needed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 42   # fixed seed so the split is reproducible

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)

## Step 1 — Getting the data

For this project I use the open-source Kaggle dataset *Heart Failure Prediction*
(Fedesoriano, 2021): **918 patients × 11 clinical features + 1 target column**. It is a single
CSV file, so no manual data collection is needed — I fetch it **programmatically with the official
Kaggle API**, authenticated with my own Kaggle access token (or a legacy `kaggle.json`), which downloads
the dataset from Kaggle's servers straight into this notebook. The file-picker in the next cell stays as an offline fallback and
loads exactly the same file downloaded by hand.

| Column | Meaning |
|---|---|
| `Age` | age of the patient (years) |
| `Sex` | M / F |
| `ChestPainType` | TA (typical angina), ATA (atypical angina), NAP (non-anginal pain), ASY (asymptomatic) |
| `RestingBP` | resting blood pressure (mm Hg) |
| `Cholesterol` | serum cholesterol (mm/dl) |
| `FastingBS` | 1 if fasting blood sugar > 120 mg/dl, else 0 |
| `RestingECG` | Normal / ST / LVH |
| `MaxHR` | maximum heart rate achieved |
| `ExerciseAngina` | exercise-induced angina: Y / N |
| `Oldpeak` | ST depression measured during exercise |
| `ST_Slope` | slope of the peak exercise ST segment: Up / Flat / Down |
| **`HeartDisease`** | **target: 1 = heart disease, 0 = normal** |


In [ ]:
# Option A (recommended): fetch heart.csv through the official Kaggle API
# Token: kaggle.com -> Settings -> API Tokens -> "Generate New Token" (KGAT_... access
# token, needs kaggle CLI >= 1.8) or a legacy kaggle.json under "Legacy API Credentials".
import os, glob, shutil, subprocess, sys, getpass

def _kaggle_credentials_ready():
    """True when a Kaggle token the API can use is already in place."""
    if os.environ.get('KAGGLE_API_TOKEN'):
        return True
    kaggle_dir = os.path.expanduser('~/.kaggle')
    return os.path.exists(os.path.join(kaggle_dir, 'access_token')) or \
           os.path.exists(os.path.join(kaggle_dir, 'kaggle.json'))

if os.path.exists('heart.csv'):
    print('heart.csv is already here — nothing to download.')
else:
    if not _kaggle_credentials_ready():
        print('One-time setup: I need my Kaggle API token.')
        print('  kaggle.com -> Settings -> API Tokens -> "Generate New Token"')
        pasted = getpass.getpass('Paste the token (access token or kaggle.json content; '
                                 'empty = upload a file instead): ').strip()
        kaggle_dir = os.path.expanduser('~/.kaggle')
        os.makedirs(kaggle_dir, exist_ok=True)
        if pasted.startswith('{'):
            with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as fh:
                fh.write(pasted)
            os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
        elif pasted:
            with open(os.path.join(kaggle_dir, 'access_token'), 'w') as fh:
                fh.write(pasted + '\n')
            os.chmod(os.path.join(kaggle_dir, 'access_token'), 0o600)
        else:
            try:
                from google.colab import files
                files.upload()               # choose kaggle.json or a token text file
                for name in glob.glob('kaggle*.json'):
                    shutil.copy(name, os.path.join(kaggle_dir, 'kaggle.json'))
                    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
                for name in glob.glob('access_token*') + glob.glob('*token*.txt'):
                    with open(name, errors='ignore') as fh:
                        value = fh.read().strip()
                    if value and ' ' not in value and not value.startswith('{'):
                        with open(os.path.join(kaggle_dir, 'access_token'), 'w') as fh:
                            fh.write(value + '\n')
                        os.chmod(os.path.join(kaggle_dir, 'access_token'), 0o600)
                        break
            except ImportError:
                print('Running locally: save the token as ~/.kaggle/access_token')
                print('(one line) or as ~/.kaggle/kaggle.json, then re-run this cell.')

    if _kaggle_credentials_ready():
        # access tokens need kaggle >= 1.8, so always update the client
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-qU', 'kaggle'], check=True)
        if not os.environ.get('KAGGLE_API_TOKEN') and \
                os.path.exists(os.path.expanduser('~/.kaggle/access_token')):
            with open(os.path.expanduser('~/.kaggle/access_token')) as fh:
                os.environ['KAGGLE_API_TOKEN'] = fh.read().strip()
        # official Kaggle API client: download dataset fedesoriano/heart-failure-prediction
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()                   # reads the access token / kaggle.json
        api.dataset_download_files('fedesoriano/heart-failure-prediction', path='.', unzip=True)
        print('Kaggle API download complete:', os.path.exists('heart.csv'))
    else:
        print('No Kaggle token found — skipping the API download.')
        print('Continue with the file-picker cell below (manual download) instead.')

    if os.path.exists('heart.csv'):
        df = pd.read_csv('heart.csv')
        print('Shape (rows, columns):', df.shape)
        df.head()


In [ ]:
# Fallback: manual upload (used only if the Kaggle API cell above was skipped)
import os

if not os.path.exists('heart.csv'):
    try:
        from google.colab import files
        uploaded = files.upload()          # choose heart.csv downloaded from Kaggle
    except ImportError:
        pass                               # running locally: put heart.csv next to this notebook

df = pd.read_csv('heart.csv')              # loaded on both paths (Kaggle download or upload)

print('Shape (rows, columns):', df.shape)
df.head()

In [ ]:
# Interactive spreadsheet viewer — a Colab-only extra; skipped gracefully in VS Code
try:
    from google.colab import sheets
    sheet = sheets.InteractiveSheet(df=df)
except ImportError:
    print('google.colab is not available (running locally, e.g. in VS Code) — '
          'use the built-in variable explorer / Data Viewer there instead.')

## Step 2 — Data cleaning and exploratory data analysis

Before training any model I check the quality of my data: **missing values**, **duplicated
rows**, data types, and the **class balance** of the target. Class balance matters to me — if
the target were heavily imbalanced, plain accuracy could look good while hiding bad
predictions, so I want to know the ratio before I trust any score.


In [ ]:
# Structure and data quality checks
df.info()

print('\nMissing values per column:')
print(df.isnull().sum())

print('\nDuplicated rows:', df.duplicated().sum())

print('\nTarget distribution (1 = heart disease, 0 = normal):')
print(df['HeartDisease'].value_counts())

In [ ]:
# Known data quirks worth reporting (found during EDA)
print('Cholesterol == 0 :', (df['Cholesterol'] == 0).sum(), 'records (likely unrecorded values)')
print('RestingBP   == 0 :', (df['RestingBP'] == 0).sum(), 'record (impossible value)')

In [ ]:
# Visual check: target balance and numeric feature distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

df['HeartDisease'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#5DA7CA', '#1A5276'])
axes[0].set_title('Target class balance')
axes[0].set_xlabel('HeartDisease (0 = normal, 1 = disease)')
axes[0].set_ylabel('Number of patients')
axes[0].tick_params(axis='x', rotation=0)

df[['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']].hist(
    ax=axes[1], color='#2E86C1', alpha=0.85)
axes[1].set_title('Numeric feature distributions')
plt.show()

In [ ]:
# Correlation between numeric features and the target (Seaborn heatmap)
plt.figure(figsize=(7.5, 6), constrained_layout=True)
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='Blues',
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation of numeric features with HeartDisease')
plt.show()

## Step 3 — Preprocessing

1. **Encoding categorical variables** — `Sex, ChestPainType, RestingECG, ExerciseAngina,
   ST_Slope` are text, but scikit-learn models need numbers. I one-hot encode them, which
   turns each category into its own 0/1 column → 20 features in total.
2. **Feature scaling** — I standardise every feature with `StandardScaler` (mean 0,
   variance 1). This is essential for **K-Nearest Neighbours**, which measures distances:
   without scaling, `Cholesterol` (up to 600) would dominate `Oldpeak` (about −2 to 6).
3. **Train/test split** — I use the classic **80% train / 20% test** split, with
   `stratify=y` so both sets keep the same disease/normal ratio, and a fixed
   `random_state` so my results are reproducible. I touch the test set **only once**,
   at evaluation time.


In [ ]:
TARGET = 'HeartDisease'

# 3.1 One-hot encoding of the 5 categorical columns
X = pd.get_dummies(df.drop(columns=[TARGET]))
y = df[TARGET]
print('Features after one-hot encoding:', X.shape[1])
print(list(X.columns))

# 3.2 80/20 stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# 3.3 Standard scaling — fit ONLY on the training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\nTraining set:', X_train_scaled.shape, '| Test set:', X_test_scaled.shape)
print('Train classes:', y_train.value_counts().to_dict())
print('Test  classes:', y_test.value_counts().to_dict())

## Step 4 — Training three supervised models

I train and compare three supervised classifiers:

| Model | Idea | Why I chose it |
|---|---|---|
| **Logistic Regression** | linear decision boundary, probability output | interpretable coefficients, strong medical baseline |
| **K-Nearest Neighbours (k=5)** | classify by the majority class of the 5 closest patients | simple, distance-based (it needs my scaling from Step 3!) |
| **Decision Tree (max_depth=3)** | human-readable if/else rules | shows non-linear patterns; I cap the depth to control **overfitting** |

**A note on overfitting:** a Decision Tree with no depth limit keeps splitting until it
memorises the training set (train accuracy 100%) but then generalises poorly to patients it
has never seen. Capping `max_depth=3` keeps the tree simple. I also print the train-test gap
for every model — a large gap is the classic warning sign of overfitting.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
}

trained = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc  = accuracy_score(y_test,  model.predict(X_test_scaled))
    trained[name] = model
    gap = train_acc - test_acc
    print(f'{name:22s} train: {train_acc:.3f} | test: {test_acc:.3f} | gap: {gap:+.3f}')

print('\nA large train-test gap is a warning sign of overfitting.')

## Step 5 — Evaluation

For a medical screening system, the four metrics are all computed from the confusion matrix:

- **Accuracy** = (tp + tn) / (tp + tn + fp + fn) — overall correctness
- **Precision** = tp / (tp + fp) — of the patients I flag as sick, how many really are
- **Recall** = tp / (tp + fn) — of the really sick patients, how many I catch
- **F1-score** = 2 · (precision · recall) / (precision + recall) — balance of the two

**Why recall is my priority:** a **false negative** (telling a sick patient they are healthy)
can delay treatment, while a false positive only leads to one extra check-up. My proposal
therefore sets two goals: **accuracy ≥ 85%** *and* the **highest possible recall**.


In [ ]:
# Comparison table of the evaluation metrics on the untouched test set
rows = []
for name, model in trained.items():
    y_pred = model.predict(X_test_scaled)
    rows.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1-score':  f1_score(y_test, y_pred),
    })

results = pd.DataFrame(rows).set_index('Model').round(3)
print(results)

best_model_name = results['F1-score'].idxmax()
best_model = trained[best_model_name]
print(f'\nBest model by F1-score: {best_model_name}')

In [ ]:
# Confusion matrices for all three models (tp / tn / fp / fn)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), constrained_layout=True)
for ax, (name, model) in zip(axes, trained.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test_scaled, y_test,
        display_labels=['Normal', 'Disease'], cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(name)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
plt.show()

# Where do the errors of the best model come from?
tn, fp, fn, tp = confusion_matrix(y_test, best_model.predict(X_test_scaled)).ravel()
print(f'Best model ({best_model_name}) on the test set:')
print(f'  True positives  (tp): {tp}   sick patients correctly flagged')
print(f'  True negatives  (tn): {tn}   healthy patients correctly cleared')
print(f'  False positives (fp): {fp}   healthy patients sent for extra checks')
print(f'  False negatives (fn): {fn}   sick patients MISSED  <- the most costly error in medicine')

In [ ]:
# Visual comparison of the three models
fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
results.plot(kind='bar', ax=ax,
             color=['#2E86C1', '#5DA7CA', '#1A5276', '#8FBEDC'])
ax.set_title('Model comparison on the test set')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
ax.tick_params(axis='x', rotation=0)
plt.show()

print(classification_report(y_test, best_model.predict(X_test_scaled),
                            target_names=['Normal', 'Disease']))

## Step 6 — Predicting the risk for a single patient

The function below applies the **exact same preprocessing** I fitted during training
(one-hot encoding → scaling → best model) to one new patient, so the system treats a new
patient exactly the way it treated the test set during evaluation.


In [ ]:
def predict_patient(age, sex, chest_pain_type, resting_bp, cholesterol, fasting_bs,
                    resting_ecg, max_hr, exercise_angina, oldpeak, st_slope):
    """Return the prediction and risk probability for one patient."""
    patient = pd.DataFrame([{
        'Age': age, 'Sex': sex, 'ChestPainType': chest_pain_type,
        'RestingBP': resting_bp, 'Cholesterol': cholesterol, 'FastingBS': fasting_bs,
        'RestingECG': resting_ecg, 'MaxHR': max_hr, 'ExerciseAngina': exercise_angina,
        'Oldpeak': oldpeak, 'ST_Slope': st_slope,
    }])
    patient_enc   = pd.get_dummies(patient).reindex(columns=X.columns, fill_value=0)
    patient_scaled = scaler.transform(patient_enc)
    pred = best_model.predict(patient_scaled)[0]
    prob = best_model.predict_proba(patient_scaled)[0][1]
    label = 'HIGH RISK of heart disease' if pred == 1 else 'LOW risk of heart disease'
    return f'Prediction : {label}\nRisk score : {prob:.1%} probability of heart disease'

# Example patient — change the values and run again!
print(predict_patient(age=54, sex='M', chest_pain_type='ASY', resting_bp=140,
                      cholesterol=239, fasting_bs=0, resting_ecg='ST', max_hr=160,
                      exercise_angina='Y', oldpeak=1.2, st_slope='Flat'))

## Step 6b — Explaining the prediction with the Google Gemini API (free tier)

My K-Nearest Neighbours model makes the decision; this step adds a communication layer on top.
After every prediction I send **only the anonymous clinical numbers and my model's result** — there
are no names or IDs to send — to the **Google Gemini API** and ask its fast `gemini-3.5-flash-lite`
model to turn them into a short, patient-friendly explanation of which values drive the risk score.
The classifier stays the brain of my system: the API can only *explain* what my own model produced,
never change it.

**Why Gemini?** I first wired this step to Z.ai's API, but their platform refused every call with
`HTTP 429: Insufficient balance or no resource package` — their API requires a paid balance. The
Gemini API has a **free tier**: a free key from **Google AI Studio** (aistudio.google.com →
"Get API key") needs no credit card and covers the fast flash models with rate limits, and my demo
makes only a handful of calls. Only this one cell changed when I switched — my model, the web app
and all results stay identical. Current model names and quotas live on the AI Studio *Models* and
*Rate limits* doc pages; the model is switchable without code edits via the `GEMINI_MODEL`
environment variable.

I built this with an **offline fallback** on purpose: if I enter no API key (or the API is
unreachable or rate-limited), the same function prints an explanation computed locally from the
data, so my system never depends on an external service to work. The key is typed into a hidden
password prompt (or provided beforehand as the `GEMINI_API_KEY` environment variable / Colab
secret) and is never printed.

In [ ]:
# API call with Python's standard library only — no extra package needed
import os, json, getpass, urllib.request

GEMINI_API_BASE = 'https://generativelanguage.googleapis.com/v1beta/models'
# Model switch without editing code, e.g. in Colab:  os.environ['GEMINI_MODEL'] = 'gemini-flash-latest'
GEMINI_MODEL = os.environ.get('GEMINI_MODEL', 'gemini-3.5-flash-lite')

def score_patient(patient_dict):
    """Apply the exact Step 6 preprocessing pipeline to a patient dict -> (prediction, probability)."""
    p = pd.DataFrame([patient_dict])
    p_enc = pd.get_dummies(p).reindex(columns=X.columns, fill_value=0)
    p_scaled = scaler.transform(p_enc)
    return best_model.predict(p_scaled)[0], best_model.predict_proba(p_scaled)[0][1]

def notable_factors(patient_dict):
    """Compare the patient's values with the healthy group of my training data."""
    healthy = df[df['HeartDisease'] == 0]
    notes = []
    for col, direction, unit, dec in [('Cholesterol', 'higher', 'mm/dl', 0), ('Oldpeak', 'higher', '', 1),
                                      ('RestingBP', 'higher', 'mm Hg', 0), ('Age', 'higher', 'years', 0),
                                      ('MaxHR', 'lower', 'bpm', 0)]:
        mean_healthy = healthy[col].mean()
        value = patient_dict[col]
        if direction == 'higher' and value > mean_healthy * 1.1:
            notes.append(f'{col}: {value:g} {unit} is well above the healthy-group average ({mean_healthy:.{dec}f})')
        if direction == 'lower' and value < mean_healthy * 0.9:
            notes.append(f'{col}: {value:g} {unit} is below the healthy-group average ({mean_healthy:.{dec}f})')
    if patient_dict.get('ExerciseAngina') == 'Y':
        notes.append('ExerciseAngina: Y (exercise-induced angina)')
    if patient_dict.get('ST_Slope') in ('Flat', 'Down'):
        notes.append(f"ST_Slope: {patient_dict['ST_Slope']} (abnormal exercise ST slope)")
    if patient_dict.get('ChestPainType') == 'ASY':
        notes.append('ChestPainType: ASY (asymptomatic — often the hardest cases for screening)')
    if patient_dict.get('FastingBS') == 1:
        notes.append('FastingBS: 1 (fasting blood sugar above 120 mg/dl)')
    return notes

def gemini_explanation(patient_dict, pred, prob, api_key):
    """Call the Google Gemini generateContent API and return the explanation text."""
    facts = '; '.join(f'{k}={v}' for k, v in patient_dict.items())
    notes = '; '.join(notable_factors(patient_dict)) or 'none strongly'
    prompt = (
        'You help a non-technical reader understand the output of a heart-disease screening prototype to a '
        'non-technical audience. The ' + best_model_name + ' model classified this anonymous patient as '
        f"{'HIGH RISK' if pred == 1 else 'LOW risk'} with a {prob:.0%} probability of heart disease. "
        f'Patient values (11 clinical features, anonymised): {facts}. '
        f'Factors that stand out against the healthy group: {notes}. '
        'Write 4-5 short plain-language sentences explaining what this result means and which of the '
        'listed values most likely influenced it. End with one sentence reminding that this is a '
        'screening tool, not a diagnosis. Use only the values listed; do not invent any facts.')
    body = json.dumps({
        'contents': [{'parts': [{'text': prompt}]}],
        'generationConfig': {'temperature': 0.7},
    }).encode()
    url = f'{GEMINI_API_BASE}/{GEMINI_MODEL}:generateContent'
    req = urllib.request.Request(url, data=body, method='POST', headers={
        'Content-Type': 'application/json',
        'x-goog-api-key': api_key,                    # auth via header, not a URL parameter
    })
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            reply = json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        detail = ''
        try:
            detail = json.loads(e.read().decode()).get('error', {}).get('message', '')
        except Exception:
            pass
        raise RuntimeError(f'HTTP {e.code}: {detail or e.reason}') from e
    candidates = reply.get('candidates') or []
    if not candidates:
        reason = (reply.get('promptFeedback') or {}).get('blockReason') or 'no candidates returned'
        raise RuntimeError(f'empty response ({reason})')
    parts = candidates[0].get('content', {}).get('parts', [])
    text = ''.join(p.get('text', '') for p in parts).strip()
    if not text:
        raise RuntimeError(f"empty response (finishReason={candidates[0].get('finishReason')})")
    return text

def explain_prediction(patient_dict, pred, prob, api_key=None):
    """Explain a prediction with the Gemini API; fall back to a locally computed explanation."""
    if api_key:
        try:
            text = gemini_explanation(patient_dict, pred, prob, api_key)
        except Exception as exc:
            print(f'Gemini API call failed ({exc.__class__.__name__}: {exc}) — '
                  'using the offline explanation instead.')
            print()
            msg = str(exc)
            if 'API key not valid' in msg or 'API_KEY_INVALID' in msg:
                print('Diagnosis: the server rejected the key itself — check that it was copied fully')
                print('and is still valid (create a new free key at aistudio.google.com).')
            elif 'HTTP 429' in msg or 'RESOURCE_EXHAUSTED' in msg or 'quota' in msg.lower():
                print('Diagnosis: the free-tier rate limit or daily quota was hit — wait a minute and')
                print('run this cell again (the free tier throttles, it does not bill), or check the')
                print('quota page in Google AI Studio.')
            elif 'HTTP 403' in msg or 'PERMISSION_DENIED' in msg:
                print('Diagnosis: this key is not allowed to call the model — check its restrictions')
                print('in Google AI Studio.')
            print()
        else:
            print(f'Explanation generated by Google {GEMINI_MODEL} (API call):')
            print()
            print(text)
            return
    notes = '; '.join(notable_factors(patient_dict)) or 'none strongly'
    label = 'HIGH RISK of heart disease' if pred == 1 else 'LOW risk of heart disease'
    print('Offline explanation (no API key / API unreachable):')
    print()
    print(f'My model classified this patient as {label} with a {prob:.1%} risk score. For orientation, '
          f"{df['HeartDisease'].mean():.0%} of the patients in my training data have heart disease. The "
          f'values that stand out against my healthy group are: {notes}. This is a screening result, not '
          'a diagnosis — a doctor should confirm it.')

# Re-run the Step 6 example patient and explain it
example = {'Age': 54, 'Sex': 'M', 'ChestPainType': 'ASY', 'RestingBP': 140,
           'Cholesterol': 239, 'FastingBS': 0, 'RestingECG': 'ST', 'MaxHR': 160,
           'ExerciseAngina': 'Y', 'Oldpeak': 1.2, 'ST_Slope': 'Flat'}

pred, prob = score_patient(example)
print('Step 6 result for the example patient:',
      'HIGH RISK of heart disease' if pred == 1 else 'LOW risk of heart disease',
      f'({prob:.1%} probability)')
print()

api_key = os.environ.get('GEMINI_API_KEY')           # optional: set as env var / Colab secret
if not api_key:
    try:                                             # headless runs have no keyboard to type into
        api_key = getpass.getpass('Gemini API key (leave empty to use the offline explanation): ') or None
    except Exception:
        api_key = None                               # no keyboard -> offline explanation

explain_prediction(example, pred, prob, api_key)

## Step 7 — Responsible AI reflection

Before such a system could ever be used in the real world, I have to be honest about its
limits and risks:

- **False negatives cost the most.** I prioritised *recall* for that reason: my best model
  still misses a few sick patients — the false-negative cell of my confusion matrix shows who
  they are (often atypical symptoms such as NAP/ATA chest pain). A real clinic would
  therefore keep a **human doctor in the loop** — this system is decision *support*, never a
  diagnosis.
- **Bias.** The dataset contains far more male than female patients (~79% vs ~21%) and no
  patients from Myanmar, so my model's accuracy may differ across groups it rarely saw.
  A deployed system would need local data and fairness checks per subgroup.
- **Privacy and data protection.** The dataset is public and anonymised (no names, IDs or
  images), and my demo collects nothing — patient values stay inside the notebook session. The two
  APIs I added are equally explicit about this: the **Kaggle API** only downloads a public dataset
  (no personal data involved), and the **Gemini API** in Step 6b sends only the anonymous 11 clinical
  numbers plus my model's output — there are no names or IDs to send. When the API path is used,
  those values do leave this session and are processed by Google's Gemini service, so I state that openly instead of
  hiding it; without a key the system runs fully offline and the explanation is computed locally.
  My API keys are typed into a hidden password prompt (or a Colab secret) and are never printed or
  saved inside the notebook file.
- **LLM explanations can be wrong (hallucinations).** The Gemini explanation in Step 6b is
  generated text, not computed logic: it can over- or under-emphasise a factor. That is why the API
  only *explains* the prediction my model already made — it can never change it — and why the
  doctor-in-the-loop rule and the disclaimer below stay unchanged.
- **Transparency and explainability.** My models are simple and inspectable: logistic
  regression coefficients and decision-tree rules can be printed and explained to a
  non-technical audience.
- **Accountability.** My system states clearly that it is an educational prototype and
  cannot replace clinical judgement (the disclaimer is shown in the demo below).


## Step 8 — Packaging the trained model for the web app (joblib)

The web app in the next step runs as its **own program** (`app.py`, started with
`streamlit run app.py`) — it cannot see the variables inside this notebook session.
So I first pack everything the app needs into **one file, `heart_model.pkl`**:

- the **best model** from Step 5, the **scaler** fitted in Step 3 and the exact
  **one-hot column list** the model expects (so `app.py` repeats my *identical*
  preprocessing — no copy-paste drift),
- the **test metrics** for the "model card" sidebar, the **healthy-group averages**
  used for the plain-language interpretation, and dataset + ownership info.

`joblib` is the standard scikit-learn companion for this (the same way my friend's
gym app ships its XGBoost model to Streamlit).

In [ ]:
# 8.1 Bundle everything the web app needs into ONE deployable file.
import os, joblib

bundle = {
    'model':         best_model,        # winning classifier from Step 5
    'scaler':        scaler,            # StandardScaler fitted on X_train (Step 3.3)
    'columns':       list(X.columns),   # exact one-hot feature order the model expects
    'model_name':    best_model_name,
    'metrics': {                        # untouched test set (Step 5) -> app sidebar
        'accuracy':  float(results.loc[best_model_name, 'Accuracy']),
        'precision': float(results.loc[best_model_name, 'Precision']),
        'recall':    float(results.loc[best_model_name, 'Recall']),
        'f1':        float(results.loc[best_model_name, 'F1-score']),
    },
    # healthy-group averages (df is my training data) -> used for the interpretation text
    'healthy_means': df[df['HeartDisease'] == 0][
        ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']].mean().round(1).to_dict(),
    'dataset': {'rows': int(len(df)),
                'disease_share': float(df['HeartDisease'].mean()),
                'source': 'Kaggle: fedesoriano/heart-failure-prediction'},
    'owner': {'name': 'Ar Kar Phyo'},
}

joblib.dump(bundle, 'heart_model.pkl')
print(f'Saved heart_model.pkl ({os.path.getsize("heart_model.pkl")/1024:.0f} KB)')
print('Packaged model :', best_model_name)
print('Test metrics   :', {k: round(v, 3) for k, v in bundle['metrics'].items()})
print('Web-app files  : app.py + heart_model.pkl + requirements.txt')

## Step 9 — The web app: Streamlit (the `localhost:8501` demo)

Instead of the old Gradio demo I now build a **Streamlit** app, the same kind of app
my classmate showed (sliders + radio buttons + a model-card sidebar, served on
`localhost:8501`). Streamlit turns **one Python file** into the whole webpage — no
HTML to write.

The next cell **writes `app.py` to disk** (`%%writefile`) instead of running it here,
because a Streamlit server has to be started from a **terminal**
(`streamlit run app.py`) — not by pressing Run in a notebook. `app.py`:

- loads `heart_model.pkl` once (`@st.cache_resource`) and rebuilds my exact
  Step 3 + Step 6 pipeline for every patient (one-hot → reindex → scale → predict),
- collects the 11 clinical inputs with **sliders / radios / dropdowns**,
- shows the **risk score, a progress bar and a plain-language interpretation**,
- carries a **model-card sidebar** (metrics from Step 5, dataset info, my name)
  and the same **educational-use disclaimer** as Step 7.

In [ ]:
%%writefile app.py
# %%writefile app.py
# Heart Disease Risk Prediction System - Streamlit web app
# Educational research prototype by Ar Kar Phyo.
#
# UI (v2): light clinical theme, wide layout, no sidebar. Three tabs:
#   1. Risk assessment - intake form (left) + risk report panel (right)
#   2. Model card      - metrics, pipeline description, author
#   3. About the data  - dataset facts, healthy-group averages, data dictionary
# Colors live in .streamlit/config.toml. Two preset buttons load demo patients
# (high-risk: 100%, healthy: 0% - verified against the packaged model).
# The prediction pipeline is EXACTLY the notebook's (Steps 3-6): one-hot encode,
# reindex to the training columns, StandardScaler, K-Nearest Neighbors.
# Run: streamlit run app.py  (or double-click start_app.bat) -> http://localhost:8501
import os
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title='Heart Disease Risk Prediction System', layout='wide')

# ---------- styling: own CSS classes only, so nothing breaks between versions ----------
CSS = '''
<style>
.block-container {max-width: 1080px; padding-top: 1.4rem;}
.hero {background: linear-gradient(130deg, #0F766E 0%, #115E59 55%, #134E4A 100%);
       border-radius: 18px; padding: 1.5rem 2rem 1.3rem 2rem; color: #F0FDFA;}
.hero h1 {margin: 0; font-weight: 800; font-size: 1.85rem; color: #F0FDFA;
          letter-spacing: .01em;}
.hero p {margin: .4rem 0 0 0; color: #99F6E4; font-size: .98rem;}
.pill {display: inline-block; background: rgba(255,255,255,.13);
       border: 1px solid rgba(255,255,255,.28); color: #CCFBF1;
       padding: .14rem .75rem; border-radius: 999px; margin: .7rem .45rem 0 0;
       font-size: .8rem;}
.badge {border-radius: 18px; padding: 1.15rem 1.3rem; text-align: center; color: #FFFFFF;}
.badge.high {background: linear-gradient(135deg, #DC2626, #B91C1C);}
.badge.low  {background: linear-gradient(135deg, #059669, #047857);}
.badge .pct {font-size: 2.5rem; font-weight: 800; line-height: 1.05;}
.badge .lbl {font-size: .95rem; opacity: .93; margin-top: .25rem;}
.chip {display: inline-block; background: #FFFBEB; border: 1px solid #FCD34D;
       color: #92400E; padding: .18rem .75rem; border-radius: 999px;
       margin: .18rem .3rem .18rem 0; font-size: .84rem;}
.note {background: #F1F5F9; border-left: 4px solid #0F766E; border-radius: 8px;
       padding: .75rem 1rem; color: #334155; font-size: .92rem;}
</style>'''
st.markdown(CSS, unsafe_allow_html=True)

# ---------- load the packaged pipeline (notebook Steps 3-6), cached for the session ----------
@st.cache_resource
def load_bundle():
    if not os.path.exists('heart_model.pkl'):
        st.error('heart_model.pkl not found - run the notebook up to Step 8 first, '
                 'then put the file next to app.py.')
        st.stop()
    return joblib.load('heart_model.pkl')

bundle = load_bundle()
model, scaler = bundle['model'], bundle['scaler']
EXPECTED_COLUMNS = bundle['columns']
m = bundle['metrics']
hm = bundle['healthy_means']

# ---------- preset patients for the demo (probabilities verified against the model) ----------
PRESET_HIGH = {'Age': 54, 'Sex': 'M', 'ChestPainType': 'ASY', 'RestingBP': 140,
               'Cholesterol': 239, 'FastingBS': 0, 'RestingECG': 'ST', 'MaxHR': 160,
               'ExerciseAngina': 'Y', 'Oldpeak': 1.2, 'ST_Slope': 'Flat'}          # -> 100%
PRESET_HEALTHY = {'Age': 38, 'Sex': 'F', 'ChestPainType': 'ATA', 'RestingBP': 120,
                  'Cholesterol': 180, 'FastingBS': 0, 'RestingECG': 'Normal',
                  'MaxHR': 175, 'ExerciseAngina': 'N', 'Oldpeak': 0.0,
                  'ST_Slope': 'Up'}                                                # -> 0%

# ---------- prediction: identical pipeline to the notebook ----------
def predict(age, sex, cpt, rbp, chol, fbs, ecg, maxhr, ang, oldpeak, slope):
    patient = pd.DataFrame([{'Age': age, 'Sex': sex, 'ChestPainType': cpt,
                             'RestingBP': rbp, 'Cholesterol': chol, 'FastingBS': fbs,
                             'RestingECG': ecg, 'MaxHR': maxhr,
                             'ExerciseAngina': ang, 'Oldpeak': oldpeak,
                             'ST_Slope': slope}])
    enc    = pd.get_dummies(patient).reindex(columns=EXPECTED_COLUMNS, fill_value=0)
    scaled = scaler.transform(enc)
    prob   = float(model.predict_proba(scaled)[0][1])
    return prob

def standout_factors(age, rbp, chol, maxhr, oldpeak, fbs, ang, slope, cpt):
    # plain-language flags, computed from the healthy-group averages in the bundle
    notes = []
    checks = [('Age', age, 1.10, 'above'), ('RestingBP', rbp, 1.10, 'above'),
              ('Cholesterol', chol, 1.10, 'above'),
              ('Oldpeak', oldpeak, 1.10, 'above'),
              ('MaxHR', maxhr, 0.90, 'below')]
    for key, value, factor, word in checks:
        if value > hm[key] * factor and word == 'above':
            notes.append(f'{key} ({value:g}) is well above the healthy-group average ({hm[key]:g})')
        if value < hm[key] * factor and word == 'below':
            notes.append(f'{key} ({value:g}) is well below the healthy-group average ({hm[key]:g})')
    if fbs == 1:
        notes.append('Fasting blood sugar above 120 mg/dl')
    if ang == 'Y':
        notes.append('Exercise-induced angina')
    if slope in ('Flat', 'Down'):
        notes.append(f'Abnormal exercise ST slope ({slope})')
    if cpt == 'ASY':
        notes.append('Asymptomatic chest pain presentation (ASY)')
    return notes

# ---------- hero banner ----------
owner, d = bundle['owner'], bundle['dataset']
st.markdown(
    '<div class="hero">'
    '<h1>Heart Disease Risk Prediction System</h1>'
    f'<p>A K-Nearest Neighbours screening assistant, trained on {d["rows"]} public patient '
    'records - enter a patient profile and see what the model would flag.</p>'
    f'<span class="pill">{owner["name"]}</span>'
    
    '<span class="pill">Educational prototype - not a medical device</span>'
    '</div>', unsafe_allow_html=True)
st.write('')

tab_predict, tab_model, tab_data = st.tabs(['Risk assessment', 'Model card', 'About the data'])

# ============================== tab 1: risk assessment ==============================
with tab_predict:
    left, right = st.columns([3, 2], gap='large')

    # ----- left: intake form -----
    with left:
        st.markdown('##### Patient intake form')
        p1, p2 = st.columns(2)
        if p1.button('Load high-risk example', use_container_width=True):
            st.session_state.update(PRESET_HIGH)
            st.rerun()
        if p2.button('Load healthy example', use_container_width=True):
            st.session_state.update(PRESET_HEALTHY)
            st.rerun()

        st.markdown('**Demographics**')
        c1, c2 = st.columns(2)
        with c1:
            age = st.slider('Age (years)', 20, 100, 54, key='Age')
        with c2:
            sex = st.radio('Sex', ['M', 'F'], horizontal=True, key='Sex')

        st.markdown('**Vitals and blood work**')
        c1, c2, c3 = st.columns(3)
        with c1:
            rbp = st.slider('Resting blood pressure (mm Hg)', 80, 200, 140, key='RestingBP')
        with c2:
            chol = st.slider('Cholesterol (mm/dl)', 0, 600, 239, key='Cholesterol')
        with c3:
            fbs = st.radio('Fasting blood sugar > 120 mg/dl', [0, 1], horizontal=True,
                           format_func=lambda v: 'No' if v == 0 else 'Yes', key='FastingBS')

        st.markdown('**Exercise and ECG findings**')
        c1, c2 = st.columns(2)
        with c1:
            maxhr = st.slider('Max heart rate (bpm)', 60, 210, 160, key='MaxHR')
            ang = st.radio('Exercise-induced angina', ['N', 'Y'], horizontal=True,
                           key='ExerciseAngina')
            oldpeak = st.slider('Oldpeak - ST depression during exercise', -3.0, 7.0, 1.2,
                                0.1, key='Oldpeak')
        with c2:
            ecg = st.selectbox('Resting ECG', ['Normal', 'ST', 'LVH'], key='RestingECG')
            cpt = st.selectbox('Chest pain type', ['ATA', 'NAP', 'ASY', 'TA'],
                               help='ATA: atypical angina, NAP: non-anginal pain, '
                                    'ASY: asymptomatic, TA: typical angina',
                               key='ChestPainType')
            slope = st.selectbox('ST slope', ['Up', 'Flat', 'Down'], key='ST_Slope')

    # ----- right: risk report -----
    with right:
        st.markdown('##### Risk report')
        clicked = st.button('Predict risk', type='primary', use_container_width=True)
        if clicked:
            prob = predict(age, sex, cpt, rbp, chol, fbs, ecg, maxhr, ang, oldpeak, slope)
            st.session_state['result'] = {'prob': prob,
                                          'notes': standout_factors(age, rbp, chol, maxhr,
                                                                    oldpeak, fbs, ang,
                                                                    slope, cpt)}
        result = st.session_state.get('result')
        if result is None:
            st.markdown('<div class="note">Fill in the patient values on the left (or load '
                        'an example), then press <b>Predict risk</b>. The result appears '
                        'here and stays until you predict again.</div>',
                        unsafe_allow_html=True)
        else:
            prob = result['prob']
            is_high = prob >= 0.5
            cls = 'high' if is_high else 'low'
            label = 'HIGH RISK of heart disease' if is_high else 'LOW risk of heart disease'
            st.markdown(
                f'<div class="badge {cls}"><div class="pct">{prob:.0%}</div>'
                f'<div class="lbl">{label}</div></div>', unsafe_allow_html=True)
            st.progress(min(max(prob, 0.0), 1.0))
            st.caption(f'The {bundle["model_name"]} decision boundary sits at 50% - this '
                       f'patient is on the {"disease" if is_high else "healthy"} side of it.')
            st.markdown('**Values that stand out against the healthy group:**')
            if result['notes']:
                chips = ''.join(f'<span class="chip">{n}</span>' for n in result['notes'])
                st.markdown(chips, unsafe_allow_html=True)
            else:
                st.caption('No single value stands out strongly against the healthy group.')
            with st.expander('How this prediction is made'):
                st.markdown(
                    f"The {bundle['model_name']} was trained in the notebook on "
                    f"{d['rows']} patients (80/20 stratified split, StandardScaler fitted on "
                    'the training set only). This app rebuilds that exact pipeline: one-hot '
                    'encoding -> column reindexing -> scaling -> prediction, using the '
                    'objects packaged in heart_model.pkl.')
                st.caption('Screening support only - a doctor must confirm any result.')

# ============================== tab 2: model card ==============================
with tab_model:
    st.markdown('##### Model card')
    k1, k2, k3, k4 = st.columns(4)
    k1.metric('Test accuracy', f"{m['accuracy']:.1%}")
    k2.metric('Precision', f"{m['precision']:.1%}")
    k3.metric('Recall', f"{m['recall']:.1%}")
    k4.metric('F1-score', f"{m['f1']:.2f}")
    st.markdown(
        f"**Algorithm** - {bundle['model_name']}, chosen in notebook Step 5 by the best "
        'F1-score against Decision Tree and Logistic Regression baselines. '
        f"**Training data** - {d['rows']} patients, {d['disease_share']:.0%} with heart "
        f"disease ({d['source']}).")
    st.markdown('**Pipeline in this app**')
    st.markdown(
        '1. the patient dict is one-hot encoded exactly like the training data\n\n'
        '2. columns are re-indexed to the training columns (missing dummies become 0)\n\n'
        '3. the StandardScaler packaged in Step 8 transforms the row\n\n'
        '4. the packaged KNN returns the probability of the disease class')
    st.markdown(f"**Author** - {owner['name']}. "
                'This web app is the communication layer of the notebook project; '
                'it never re-trains anything.')

# ============================== tab 3: about the data ==============================
with tab_data:
    st.markdown('##### About the data')
    st.markdown(
        f"The model was trained on {d['rows']} patient records published on Kaggle "
        f"({d['source']}); {d['disease_share']:.0%} of them have heart disease. "
        'The averages below come from the healthy group of that dataset and are what '
        'the risk report compares your patient against.')
    hm_table = pd.DataFrame({'Feature': list(hm.keys()),
                             'Healthy-group average': [f'{v:g}' for v in hm.values()]})
    st.dataframe(hm_table, hide_index=True, use_container_width=True)
    st.markdown('##### The 11 input features')
    dictionary = pd.DataFrame([
        ('Age', 'years of the patient'),
        ('Sex', 'M = male, F = female'),
        ('ChestPainType', 'ATA = atypical angina, NAP = non-anginal pain, '
         'ASY = asymptomatic, TA = typical angina'),
        ('RestingBP', 'resting blood pressure in mm Hg'),
        ('Cholesterol', 'serum cholesterol in mm/dl'),
        ('FastingBS', '1 if fasting blood sugar > 120 mg/dl, else 0'),
        ('RestingECG', 'Normal, ST (ST-T abnormality), LVH (left ventricular hypertrophy)'),
        ('MaxHR', 'maximum heart rate achieved, in bpm'),
        ('ExerciseAngina', 'Y = angina induced by exercise, N = no'),
        ('Oldpeak', 'ST depression measured during exercise relative to rest'),
        ('ST_Slope', 'the slope of the peak exercise ST segment: Up, Flat, Down'),
    ], columns=['Feature', 'Meaning'])
    st.dataframe(dictionary, hide_index=True, use_container_width=True)

st.caption('Educational prototype - not a medical device. Predictions do not replace '
           'a doctor.')


In [ ]:
%%writefile requirements.txt
# Pinned to the exact versions tested on this project (Windows, Python 3.14).
# Install once:  python -m pip install -r requirements.txt
streamlit==1.67.0
pandas==3.1.1
numpy==2.5.5
scikit-learn==1.9.1
joblib==1.6.0


In [ ]:
# 9.3 Sanity check before shipping: app.py must be valid Python and all files present.
import ast, os

ast.parse(open('app.py').read())
print('app.py compiles - syntax OK')
for name in ['app.py', 'requirements.txt', 'heart_model.pkl']:
    state = 'OK' if os.path.exists(name) else 'MISSING'
    print(f'{name:18s} {state:8s} {os.path.getsize(name)/1024:8.1f} KB')

### How to run the app from this repository

Everything the app needs is generated inside this notebook: Step 8 writes
`heart_model.pkl`, and Step 9 writes `app.py` and `requirements.txt` — all three
land in the same folder as this notebook, so after one full run nothing else has
to be carried over. Here is how to run my project on a Windows machine with
Anaconda installed:

**Option A — double-click `start_app.bat` (simplest):**

Open the project folder and double-click `start_app.bat`. It checks
that the required packages are installed, rebuilds `heart_model.pkl` with the
machine's own scikit-learn version, and starts the server. When the console
shows the URL, the app is live — no commands to remember.

**Option B — VS Code / terminal (the way I ran it during development):**

1. Open the folder containing `app.py`, `heart_model.pkl` and `requirements.txt`
   in VS Code.
2. Select the Anaconda interpreter (`Ctrl+Shift+P` → *Python: Select Interpreter*).
3. In the VS Code terminal:
   ```bash
   pip install -r requirements.txt
   streamlit run app.py
   ```
   If Windows says `streamlit is not recognized`, the install still worked — pip
   placed streamlit.exe outside PATH. Run it as a module instead:
   `python -m streamlit run app.py` (same app, no PATH changes needed).
4. The browser opens at **http://localhost:8501**. The **Risk assessment** tab
   holds the intake form — pressing *Load high-risk example* predicts 100
   percent, *Load healthy example* predicts 0 percent, and the report panel
   lists the factors behind each verdict. The **Model card** tab shows the test
   metrics, and **About the data** shows the dataset facts and data dictionary.

**Google Colab (quick temporary check only):**
```python
!pip install -q -r requirements.txt
!streamlit run app.py & npx localtunnel --port 8501
```
Colab runs on a server, so there is no `localhost` in your browser — the link
goes through a tunnel (paste the public IP it shows as the tunnel password).
The URL dies when the session stops, which is why sharing with reviewers should use Option A
or B.

**A permanent public link (optional):** pushing `app.py`, `heart_model.pkl` and
`requirements.txt` to GitHub and deploying on **Streamlit Community Cloud**
(share.streamlit.io) gives a free, stable URL that stays reachable after this
notebook is closed.
